# Phase 4 — Destination Research

**Main objective**: add a specific branch to the graph designed to help users find the best destination based on their requirements. An initial router determines whether this branch should be executed, depending on whether the user has already specified a destination.

Once the router is executed, the branch will:

* Generate a shortlist of candidate destinations.
* Use the available MCPs to research each candidate **only when needed**. If the llm can answer using its own general knowledge, we avoid using MCPs' tools to reduce costs associated with the tokens they generate.
* Score and compare candidate destinations.
* Give a ranking to each destination based on its relevance and supporting evidence.

In [1]:
# Imports and environment setup

import os
import json
import uuid
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv
from IPython.display import Image, display, Markdown
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, ToolMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

## *Code inherited from phase3.ipynb*

In [2]:
def sanitize_messages_for_llm(messages):
    """
    Auxiliary function for removing file/resource blocks from ToolMessages before 
    sending them to OpenAI Chat Completions.

    This fixes the error "OpenAI Chat Completions does not support file URLs." that 
    occured passing the trvl tool's response to the LLM.

    The original messages remain unchanged in LangGraph state.
    """

    sanitized = []

    for message in messages:

        if isinstance(message, ToolMessage) and isinstance(message.content, list):

            text_blocks = []

            for block in message.content:
                if (
                    isinstance(block, dict)
                    and block.get("type") == "text"
                    and "text" in block
                ):
                    text_blocks.append(block["text"])

            if text_blocks:
                sanitized.append(
                    ToolMessage(
                        content="\n".join(text_blocks),
                        tool_call_id=message.tool_call_id,
                        name=message.name,
                    )
                )

            else:
                sanitized.append(message)

        else:
            sanitized.append(message)

    return sanitized

## Implement the research router

The router is implemented as an LLM call that returns only whether or not destination search mode should be enabled. Since the task is simple in this case, a **lighter model** is used to reduce costs and improve response time.

In [3]:
# Define the shared TravelState
class TravelRequirements(BaseModel):
    origin: str | None = None
    destination: str | None = None
    travel_dates: str | None = None
    duration_days: int | None = None
    travelers: int | None = None
    budget: float | None = None
    currency: str | None = None
    interests: list[str] = Field(default_factory=list)
    constraints: list[str] = Field(default_factory=list)

class TravelState(TypedDict):
    messages: Annotated[list, add_messages]
    requirements: TravelRequirements | None
    destination_research_needed: bool # <-- New field

In [4]:
class ResearchDecision(BaseModel):
    needs_research: bool

In [ ]:
router_llm = ChatOpenAI(
    model="google/gemma-3-12b-it",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [13]:
RESEARCH_ROUTER_PROMPT = """
You are a travel-planning router.

Decide whether destination research is needed.

Set needs_research=true when the user has not committed to a single destination
and would benefit from discovering, comparing, or selecting destinations.
This includes requests that provide multiple destinations and ask to compare
or choose between them.

Set needs_research=false only when the user has clearly chosen one specific
destination, even if they want to research hotels, flights, activities, or
other options there.

Base the decision only on the user's request. Do not invent requirements.
"""

In [14]:
structured_router_llm = router_llm.with_structured_output(ResearchDecision)

async def research_router(state: TravelState):
    response = await structured_router_llm.ainvoke([
        SystemMessage(content=RESEARCH_ROUTER_PROMPT),
        *sanitize_messages_for_llm(state["messages"]),
    ])

    return {"destination_research_needed": response.needs_research}

### Check if the router decides correctly

In [15]:
async def test_router(prompt: str):
    state = {
        "messages": [HumanMessage(content=prompt)],
        "requirements": None,
        "destination_research_needed": False,
    }

    result = await research_router(state)

    print(f"Prompt: {prompt}")
    print(f"needs_research: {result['destination_research_needed']}")

In [16]:
await test_router(
    "I want to travel somewhere in Europe in September. "
    "Find me some good destinations for food and culture."
) # True

await test_router(
    "I'm going to Paris in September. Find me suitable hotels for two people."
) # False

await test_router(
    "Compare Paris, Rome and Barcelona for a week-long trip in September "
    "and tell me which one is best."
) # True

Prompt: I want to travel somewhere in Europe in September. Find me some good destinations for food and culture.
needs_research: True
Prompt: I'm going to Paris in September. Find me suitable hotels for two people.
needs_research: False
Prompt: Compare Paris, Rome and Barcelona for a week-long trip in September and tell me which one is best.
needs_research: True


## Add new conditional edge to the graph

In [ ]:
def route_destination_research(state: TravelState):
    if state["destination_research_needed"]:
        return "discovery"
    return "llm"

In [ ]:
builder.add_node("router", research_router)
builder.add_node("discovery", destination_discovery)
builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route_destination_research,
    {
        "discovery": "discovery",
        "llm": "llm",
    },
)

builder.add_edge("discovery", "llm")

builder.add_conditional_edges("llm", tools_condition)
builder.add_edge("tools", "llm")

In [ ]:
async def destination_discovery(state):
    ...
    return {
        "destination_candidates": [
            "Lisbon",
            "Rome",
            "Athens",
            "Barcelona",
        ]
    }